# Confronto di modelli di sentiment analysis sui commenti Instagram

Questo notebook **coordina l'esperimento**: richiama le funzioni dei file in `src/` (`scraping`, `preprocessing`, `models`, `evaluate`) nell'ordine giusto e mostra tabelle e grafici.

**Passi dell'esperimento**

| # | Sezione | Cosa succede |
|---|---|---|
| 0 | Configurazione | parametri e librerie |
| 1 | Raccolta dei commenti | scraping da qui (`RUN_SCRAPING = True`) o da terminale; poi riassume cosa c'è in `data/raw/` |
| 2 | Preparazione dei dati | pulizia, pseudonimi, due versioni del testo (emoji grezze / tradotte) |
| 3 | Etichettatura a mano | crea i set da etichettare (00, 01, ...) e controlla l'avanzamento: **passo manuale** |
| 4 | Dataset etichettato | unisce le etichette ai commenti |
| 5 | Modelli | esegue i modelli su entrambe le versioni del testo: **passo lungo** |
| 6 | Valutazione | metriche, grafici, test statistici, analisi degli errori |
| 7 | Salvataggio e ambiente | tabelle e grafici in `data/results/`, versioni delle librerie |
| 8 | Conclusioni | considerazioni |

**Come usarlo.** Eseguire le celle dall'alto verso il basso. Le sezioni 4, 5 e 6 funzionano solo dopo aver completato le etichette (sezione 3). I passi lunghi si salvano su file, quindi alla riapertura del notebook non si rifà tutto da capo.

**Attenzione: privacy.** Alcuni output mostrano il testo di commenti scritti da altre persone. Gli username sono già sostituiti da pseudonimi (`user_001`, ...), ma prima di fare `git commit` svuotare gli output (*Clear All Outputs* in VS Code), oppure installare `nbstripout` (`pip install nbstripout`, poi `nbstripout --install`), che lo fa da solo a ogni commit.

## 0. Configurazione

**Attenzione. Nel caso non riconosca alcune librerie:**

Esegueire in ordine:

- `source venv/bin/activate`
- `python -m pip install --upgrade pip`
- `python -m pip install -r requirements.txt` (o file contenente lista delle librerie usate)
- `python -m ipykernel install --user --name sentimentproject --display-name "Python (SentimentProject)"` (nome progetto e poi nome visivo)

In [ ]:
# Parametri dell'esperimento: modificare solo questi valori.
N_TO_LABEL = 300        # quanti commenti estrarre quando crei un nuovo set da etichettare
SEED = 42               # seme del campionamento (stesso seme = stesso campione)
NEW_SET = False         # True = crea un nuovo set da etichettare (numero successivo: 00, 01, ...)  <-----------------
SETS = None             # set di etichette da usare: None = tutti, oppure ad esempio ["00", "01"]
RERUN_MODELS = False    # True = rifai girare i modelli anche se le predizioni esistono già
N_ERRORS = 10           # quanti commenti sbagliati mostrare nell'analisi degli errori

# Scraping (sezione 1): lasciare RUN_SCRAPING = False tranne quando si vuole raccogliere commenti.
RUN_SCRAPING = False                 # True = esegui lo scraping da qui (si apre Chrome).  <-------------------------- rimettere a False
SCRAPE_PROFILE = "will_ita"         # username del profilo da leggere, senza @
SCRAPE_POSTS = 10                   # quanti post leggere
SCRAPE_COMMENTS = 100               # massimo di commenti da leggere per ogni post
RESET_SESSION = False               # True = cancella i cookie salvati e rifà il login

In [ ]:
# Ricarica da sola i file di src/ quando li modifichi, senza riavviare il notebook.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Trova la cartella principale del progetto, sia che il notebook parta da notebooks/ sia dalla radice.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import preprocessing as pp
from src import evaluate as ev

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

print("Cartella del progetto:", ROOT)

## 1. Raccolta dei commenti

si possono raccogliere i commenti **da qui** (mettendo `RUN_SCRAPING = True` nella configurazione) oppure da terminale (`python src/scraping.py`, **LEGGERE** il file *README.md* per sapere come fare): il risultato è lo stesso e finisce in `data/raw/`.

**Le credenziali di Instagram non passano mai dal notebook.** Username e password restano nel file `.env`: li legge `src/scraping.py` dentro la funzione `scrape_profile`, che li usa solo per il login e non li stampa, non li restituisce e non li salva. Per questo non compaiono né nel codice né negli output del notebook. Per tenerli al sicuro:

- non bisogna scrivere mai username e password in una cella e non stampare `os.environ`;
- non si deve mostrare il contenuto dei dati grezzi (contengono gli username **veri** dei commentatori): la cella sotto stampa solo i conteggi;
- `.env` e `data/` (cookie compresi) restano fuori da git: controlla il `.gitignore`.

**Cose da sapere**

- Si apre una finestra di Chrome: meglio non muovere il mouse sopra di essa mentre legge i post. La cella resta in esecuzione finché non ha finito (anche diversi minuti).
- Al primo avvio fa il login e salva i cookie in `data/session/`; dalle volte successive riusa la sessione. Attenzione a lanciare lo scraping molte volte di fila: troppi login fanno bloccare l'account.
- Per lo stesso profilo i due CSV in `data/raw/` vengono **sovrascritti**. Dopo lo scraping (nel notebook) rimettere `RUN_SCRAPING = False`, così in caso di esecuzione di *Run All* non lo ripete.

In [ ]:
if RUN_SCRAPING:
    if SCRAPE_PROFILE == "NOME_PROFILO":
        raise ValueError("Imposta SCRAPE_PROFILE con l'username del profilo (senza @) nella configurazione.")

    from src import scraping as sc

    posts_df, comments_df = sc.scrape_profile(SCRAPE_PROFILE, SCRAPE_POSTS, SCRAPE_COMMENTS,
                                              reset_session=RESET_SESSION)
    print(f"Raccolti {len(posts_df)} post e {len(comments_df)} commenti. \nOra rimettere RUN_SCRAPING = False.")
    del posts_df, comments_df    # i commenti contengono gli username veri: non lasciarli nel notebook
else:
    print("Scraping non eseguito (RUN_SCRAPING = False): uso i file già presenti in data/raw/.")

In [ ]:
rows = []
for comments_file in sorted(pp.RAW_DIR.glob("commenti_*.csv")):
    profile = comments_file.stem.removeprefix("commenti_")
    comments = pd.read_csv(comments_file)
    posts_file = pp.RAW_DIR / f"posts_{profile}.csv"
    posts = pd.read_csv(posts_file) if posts_file.exists() else None
    rows.append({
        "profilo": profile,
        "post": len(posts) if posts is not None else None,
        "commenti dichiarati": int(posts["comments"].sum()) if posts is not None else None,
        "commenti raccolti": len(comments),
    })

if rows:
    collected = pd.DataFrame(rows)
    display(collected)
    print("Totale commenti raccolti:", collected["commenti raccolti"].sum())
else:
    print("Nessun file commenti_*.csv in data/raw/: lancia prima lo scraping dal terminale.")

## 2. Preparazione dei dati

`prepare_comments` toglie righe incomplete e doppioni, sostituisce gli username con pseudonimi, rende anonime le `@menzioni` e crea **due versioni dello stesso testo**:

- `text_clean`: il commento così com'è estratto, con le amoji ma NON tradotte in italiano;
- `text_emoji`: lo stesso commento con le emoji tradotte in parole italiane.

Ogni modello viene valutato su entrambe. Il dataset non contiene username veri né il testo originale: la mappa degli username (`username_map`) serve solo privatamente e non va mostrata né salvata nel repository.

In [ ]:
raw = pp.load_raw_comments()
dataset, username_map = pp.prepare_comments(raw)

print(f"Commenti grezzi: {len(raw)} -> dopo la preparazione: {len(dataset)}")
display(dataset.drop(columns=["url_post"]).head(len(raw)))  # stampa illustrativa di tutti i commenti posseduti

In [ ]:
summary = pd.Series({
    "commenti": len(dataset),
    "utenti distinti": dataset["user"].nunique(),
    "post distinti": dataset["url_post"].nunique(),
    "commenti con emoji": int((dataset["text_clean"] != dataset["text_emoji"]).sum()),
    "commenti di sole emoji": int(dataset["emoji_only"].sum()),
    "lunghezza media (arrotondata)": round(dataset["text_clean"].str.len().mean()),
})
display(summary.to_frame("valore"))

# Se pochi utenti scrivono molti commenti, il campione rappresenta soprattutto loro.
per_user = dataset["user"].value_counts()
top5_sum = per_user.head(5).sum()
top5_share = top5_sum / len(dataset)
print(f"I 5 utenti più attivi scrivono il {top5_share:.0%} ({top5_sum}) dei commenti "
      f"(il massimo per un singolo utente è {per_user.iloc[0]}).")

In [ ]:
# Calcola la lunghezza di ogni commento in caratteri
lengths = dataset["text_clean"].fillna("").astype(str).str.len()

# Crea due grafici affiancati
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Grafico 1: distribuzione completa di tutti i commenti
axes[0].hist(lengths, bins=250)
axes[0].set_title("Lunghezza di tutti i commenti (0-2200)")
axes[0].set_xlabel("Caratteri")
axes[0].set_ylabel("Frequenza")

# Grafico 2: ingrandimento dei commenti più lunghi
zoom_min = 1000
zoom_max = 2200

# Definisce intervalli di 20 caratteri per rendere visibili le frequenze
bins_zoom = range(zoom_min, zoom_max + 21, 20)

axes[1].hist(lengths, bins=bins_zoom)
axes[1].set_xlim(zoom_min, zoom_max)
axes[1].set_title("Ingrandimento dei commenti più lunghi (1000-2200)")
axes[1].set_xlabel("Caratteri")
axes[1].set_ylabel("Frequenza")

# Mostra solo valori interi sull'asse Y
from matplotlib.ticker import MaxNLocator

for ax in axes:
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

# Migliora la disposizione dei grafici
plt.tight_layout()
plt.show()

## 3. Etichettatura a mano

Per sapere se un modello sbaglia serve un riferimento: le **tue** etichette. I commenti da etichettare si estraggono a **set numerati**: `data/labeled/etichettati_a_mano_00.csv`, `_01.csv`, `_02.csv`... Se ne possono creare quanti se ne vogliono, in momenti diversi (per esempio 300 commenti oggi e altri 150 dopo un nuovo scraping): ogni set contiene commenti **diversi** da quelli degli altri e non viene mai sovrascritto, quindi non si perdono le etichette già scritte. Ogni file ha tre colonne: `comment_id`, `text_clean`, `label_true`.

**Come fare**

1. Per creare un set mettere `NEW_SET = True` nella configurazione (sezione 0), eseguire la prima cella qui sotto e poi rimettere `NEW_SET = False`. (Da terminale: `python src/preprocessing.py`.)
2. Aprire il CSV e compilare `label_true` con `positivo`, `negativo` o `neutro` (maiuscole e spazi non contano). Salvare restando in formato CSV (va bene sia la virgola sia il punto e virgola).
3. Eseguire la cella di controllo: mostra, per ogni set, quanti commenti mancano ancora.

**Regole di etichettatura.** Leggere `docs/linee_guida_etichettatura.md`. Il principio è etichettare **ogni commento da solo, senza contesto**, come fanno i modelli, che ricevono solo il testo; nel file ci sono regole per insulti, ironia, domande, sole emoji, menzioni e casi misti.

**[OPZIONALE]** Fare etichettare a un'altra persona un sottoinsieme di un set (50-100 commenti, stesso formato) e salvarlo come `data/labeled/etichette_seconda_persona_NN.csv`, dove NN è il numero del set di partenza. L'ultima cella di questa sezione (terza) calcola l'accordo tra i due (kappa di Cohen), che mostra quanto le etichette sono affidabili. Orientativamente: <0.00 nessun accordo, 0.00-0.20 accordo insignificante, 0.21-0.40 accordo scarso, 0.41-0.60 accordo moderato, 0.61-0.80 sostanziale, oltre 0.80 quasi perfetto. Se le classi sono molto sbilanciate il kappa può risultare basso anche con un accordo semplice alto: riporta entrambi i numeri.

In [ ]:
existing_sets = pp.list_label_sets()

if NEW_SET:
    path, sample = pp.create_label_set(dataset, N_TO_LABEL, SEED)
    print(f"Creato {path.name} con {len(sample)} commenti: compila la colonna label_true. "
          f"Poi rimetti NEW_SET = False.")
elif existing_sets:
    print("Set già presenti:", ", ".join(number for number, _ in existing_sets),
          "(per crearne un altro metti NEW_SET = True).")
else:
    print("Nessun set di etichette: metti NEW_SET = True nella configurazione e rilancia questa cella.")

In [ ]:
# Controllo di avanzamento: rilancia questa cella dopo aver salvato le etichette.
rows = []
for number, path in pp.list_label_sets():
    part = pd.read_csv(path, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
    done = part["label_true"].fillna("").str.strip() != ""
    rows.append({"set": number, "commenti": len(part), "etichettati": int(done.sum()), "mancanti": int((~done).sum())})

progress = pd.DataFrame(rows, columns=["set", "commenti", "etichettati", "mancanti"])
display(progress)
if progress["mancanti"].sum() > 0:
    print("Mancano ancora delle etichette: completale prima di passare alla sezione 4.")

In [ ]:
# Facoltativo: accordo con una seconda persona (kappa di Cohen), per i set che hanno un file
# etichette_seconda_persona_NN.csv.
pairs = []
for number, path in pp.list_label_sets():
    second_file = pp.LABELED_DIR / f"etichette_seconda_persona_{number}.csv"
    if second_file.exists():
        first = pd.read_csv(path, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
        second = pd.read_csv(second_file, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
        pairs.append(first[["comment_id", "label_true"]].merge(
            second[["comment_id", "label_true"]], on="comment_id", suffixes=("_tu", "_altra_persona")))

if pairs:
    from sklearn.metrics import cohen_kappa_score

    both = pd.concat(pairs, ignore_index=True).dropna()
    for column in ("label_true_tu", "label_true_altra_persona"):
        both[column] = both[column].str.strip().str.lower()
    both = both[(both["label_true_tu"] != "") & (both["label_true_altra_persona"] != "")]

    agreement = (both["label_true_tu"] == both["label_true_altra_persona"]).mean()
    kappa = cohen_kappa_score(both["label_true_tu"], both["label_true_altra_persona"])
    print(f"Commenti etichettati da entrambi: {len(both)}")
    print(f"Accordo semplice: {agreement:.0%} | kappa di Cohen: {kappa:.2f}")
else:
    print("Nessun file etichette_seconda_persona_NN.csv: passaggio facoltativo, salto.")

## 4. Dataset etichettato

`attach_labels` unisce le etichette ai commenti tramite `comment_id`. Usa i set indicati in `SETS` (di default **tutti**). Se qualche riga non ha etichetta, o ne ha una diversa da `positivo` / `negativo` / `neutro`, si ferma e indica file e righe da correggere (numerate come nel foglio di calcolo, intestazione = riga 1).

In [ ]:
labeled = pp.attach_labels(dataset, pp.resolve_label_sets(SETS))

counts = labeled["label_true"].value_counts().reindex(ev.LABELS, fill_value=0)
display(pd.DataFrame({"commenti": counts, "percentuale": (counts / counts.sum() * 100).round(1)}))

# Quanti commenti per set e per etichetta.
display(pd.crosstab(labeled["set"], labeled["label_true"]).reindex(columns=list(ev.LABELS), fill_value=0))

with_emoji = int((labeled["text_clean"] != labeled["text_emoji"]).sum())
print(f"Commenti etichettati: {len(labeled)} | con emoji: {with_emoji} | di sole emoji: {int(labeled['emoji_only'].sum())}")

if counts["neutro"] == 0:
    print("Nessun commento 'neutro': il confronto sarà a due classi.")
elif counts.min() < 15:
    print("Una classe ha meno di 15 commenti: le sue metriche saranno poco affidabili.")

counts.plot.pie(autopct="%1.1f%%", startangle=90, figsize=(5, 5), title="Distribuzione delle etichette")
plt.show()

## 5. Modelli

`run_all` esegue ogni modello su `text_clean` e su `text_emoji` e salva le predizioni in `data/predictions/predizioni.csv` (dopo ogni modello, quindi un errore a metà non fa perdere quanto già calcolato).

- Il **primo avvio è lento**: ogni modello viene scaricato da Hugging Face (serve internet).
- Se le predizioni esistono già la cella le **carica senza rifare i calcoli**; per ripetere tutto metti `RERUN_MODELS = True` nella cella di configurazione.
- Per controllare che i modelli funzionino su poche frasi, dal terminale: `python src/models.py`.

In [ ]:
from src import models as md

if md.PREDICTIONS_PATH.exists() and not RERUN_MODELS:
    predictions = ev.load_predictions(md.PREDICTIONS_PATH)
    print(f"Predizioni già presenti, caricate da {md.PREDICTIONS_PATH.name} (RERUN_MODELS = False).")
else:
    # Per usare la soglia di feel_it (le sue predizioni poco sicure diventano "neutro"):
    # predictions = md.run_all(labeled, md.build_models(neutral_threshold=0.7), out_path=md.PREDICTIONS_PATH)
    predictions = md.run_all(labeled, out_path=md.PREDICTIONS_PATH)

display(predictions.groupby(["model", "version"]).size().rename("predizioni").to_frame())

missing = set(labeled["comment_id"]) - set(predictions["comment_id"])
if missing:
    print(f"Attenzione: {len(missing)} commenti etichettati non hanno predizioni: "
          f"rifai girare i modelli con RERUN_MODELS = True.")

## 6. Valutazione

`run_evaluation` confronta le predizioni con le etichette a mano. Per leggere le tabelle:

- **Macro-F1**: media delle F1 delle classi, con lo stesso peso ciascuna. È la metrica principale perché non risente dello sbilanciamento tra classi. L'**accuracy** è la quota di predizioni giuste.
- **Sottoinsiemi**: `tutti`; `con_emoji` (gli unici commenti dove le due versioni del testo differiscono); `solo_emoji`; `senza_neutro` (il confronto equo con feel_it, che sceglie solo tra positivo e negativo e non può mai dire "neutro"). Un sottoinsieme uguale a `tutti` o vuoto non compare.
- **Test di McNemar**: dice se una differenza tra due serie di risposte è probabilmente reale (`p_value` sotto 0.05) o può essere dovuta al caso. Nel confronto tra modelli guarda `p_bonferroni`, che corregge per i confronti multipli.
- Con **pochi commenti** (meno di 30) le percentuali oscillano molto: le tabelle lo segnalano.

**[!]** leggere **`README.md`** per comprendere al meglio il significato delle metriche calcolate.

### 6.1 Metriche per modello, versione del testo e sottoinsieme

In [ ]:
results = ev.run_evaluation(labeled, predictions)
metrics = results["metrics"]

for subset, table in metrics.groupby("subset", sort=False):
    n = int(table["n"].max())
    print(f"\n=== {subset} (n = {n}) ===")
    if n < ev.MIN_RELIABLE:
        print(f"attenzione: meno di {ev.MIN_RELIABLE} commenti, risultati poco affidabili")
    display(table.drop(columns="subset").round(3).reset_index(drop=True))

no_neutral = [m for m, g in results["merged"].groupby("model", sort=False) if not (g["label_pred"] == "neutro").any()]
if no_neutral and (labeled["label_true"] == "neutro").any():
    print(f"\nNota: {', '.join(no_neutral)} non predice mai 'neutro': per un confronto equo guardare 'senza_neutro'.")

### 6.2 Quale modello è migliore?

In [ ]:
for subset in ("tutti", "con_emoji", "solo_emoji", "senza_neutro"):
    part = metrics[metrics["subset"] == subset]
    if part.empty:
        continue
    ev.plot_metric(part, "macro_f1", f"Macro-F1 per modello e versione del testo ({subset})")
    plt.show()

best = metrics[metrics["subset"] == "tutti"].sort_values("macro_f1", ascending=False).iloc[0]
print(f"Miglior macro-F1 su tutti i commenti: [{best['model']}] con {best['version']} ({best['macro_f1']:.3f})\n")

best = metrics[metrics["subset"] == "con_emoji"].sort_values("macro_f1", ascending=False).iloc[0]
print(f"Miglior macro-F1 su tutti i commenti contenenti ALMENO una emoji: [{best['model']}] con {best['version']} ({best['macro_f1']:.3f})\n")

best = metrics[metrics["subset"] == "solo_emoji"].sort_values("macro_f1", ascending=False).iloc[0]
print(f"Miglior macro-F1 su tutti i commenti con SOLO emoji: [{best['model']}] con {best['version']} ({best['macro_f1']:.3f})\n")

best = metrics[metrics["subset"] == "senza_neutro"].sort_values("macro_f1", ascending=False).iloc[0]
print(f"Miglior macro-F1 su tutti i commenti senza 'neutro': [{best['model']}] con {best['version']} ({best['macro_f1']:.3f})")

### 6.3 Matrici di confusione

Righe = l'etichetta scritta a mano, colonne = la predizione del modello. Sulla diagonale ci sono i commenti giusti; fuori diagonale si vede **quali classi vengono scambiate** (per esempio positivo scambiato per neutro).

In [ ]:
ev.plot_confusions(results["confusions"])
plt.show()

### 6.4 Tradurre le emoji aiuta?

In [ ]:
# Per ogni modello: macro-F1 con le emoji grezze e con le emoji tradotte, e la differenza.
order = list(dict.fromkeys(metrics["model"]))
tutti = metrics[metrics["subset"] == "tutti"].pivot(index="model", columns="version", values="macro_f1")

effect = pd.DataFrame({
    "macro-F1, emoji grezze": tutti[ev.VERSION_CLEAN],
    "macro-F1, emoji tradotte": tutti[ev.VERSION_EMOJI],
}).reindex(order)
effect["differenza (tradotte - grezze)"] = effect["macro-F1, emoji tradotte"] - effect["macro-F1, emoji grezze"]
display(effect.round(3))

# McNemar: la differenza è probabilmente reale (p_value < 0.05) o dovuta al caso?
display(results["versions"].round(3))   # seconda tabella

`solo_text_clean` e `solo_text_emoji` contano i commenti sbagliati con una versione e giusti con l'altra: sono questi, e solo questi, a decidere il test. Se i due numeri sono piccoli o simili, la differenza tra le versioni non è dimostrabile con questi dati.

### 6.5 Un modello è davvero migliore di un altro?

In [ ]:
# Confronto a coppie con McNemar: guarda p_bonferroni (corretto per i confronti multipli).
display(results["models"].round(3))

### 6.6 Analisi degli errori

Leggi gli errori cercando **schemi ricorrenti**: sarcasmo e ironia, negazioni, emoji ambigue, gergo, commenti fatti solo di menzioni, frasi che parlano di un argomento senza esprimere un'opinione. Annota i casi più istruttivi: sono la parte qualitativa del confronto. Per guardare un altro modello cambiare `INSPECT_MODEL` e `INSPECT_VERSION`.

In [ ]:
# Di default: il modello e la versione migliori per macro-F1 (vedi 6.2).
for model in ("feel_it", "bert_italiano", "xlmr_twitter", "nlptown_stelle"):
    for version in ("text_clean", "text_emoji"):
        INSPECT_MODEL, INSPECT_VERSION = model, version

        examples = ev.error_examples(results["merged"], INSPECT_MODEL, INSPECT_VERSION, n=N_ERRORS)
        print(f"\nCommenti sbagliati da [{INSPECT_MODEL}] su {INSPECT_VERSION} ({len(examples)} esempi scelti a caso):")
        display(examples)

### 6.7 Commento per commento

Le predizioni salvate da `models.py` sono codici (`comment_id`) ed etichette: servono alle macchine. Questa tabella le rimette insieme al **testo** del commento, alla **etichetta scritta a mano** e alla risposta di ogni modello su entrambe le versioni del testo, così si legge come un foglio di lavoro. I commenti sono ordinati dai più difficili (quelli su cui i modelli sbagliano di più) ai più facili; `giuste` e `totali` dicono quante predizioni su quante sono corrette per quel commento.

Guarda in particolare i commenti su cui **tutte** le predizioni sono sbagliate: o sono davvero ambigui, oppure l'etichetta a mano va ricontrollata con le linee guida. La sezione 7 salva la tabella anche in CSV, apribile con Excel o Numbers.

In [ ]:
readable = results["readable"]

summary = pd.Series({
    "tutte le predizioni giuste": int((readable["giuste"] == readable["totali"]).sum()),
    "tutte sbagliate": int((readable["giuste"] == 0).sum()),
    "miste": int(((readable["giuste"] > 0) & (readable["giuste"] < readable["totali"])).sum()),
})
display(summary.to_frame("commenti"))

# I commenti più discordanti.
discordanti = readable[readable["giuste"] == 0]
display(discordanti)

## 7. Salvataggio e ambiente

`save_results` scrive in `data/results/` (cartella ignorata da git) tabelle e grafici da usare nella tesi: `metriche.csv`, `mcnemar_versioni.csv`, `mcnemar_modelli.csv`, `confusioni.csv`, `predizioni_leggibili.csv` (commento per commento, con il testo), `macro_f1.png`, `matrici_confusione.png`. I file precedenti vengono sovrascritti.

In [ ]:
ev.save_results(results)

In [ ]:
# Versioni delle librerie.
from importlib.metadata import version

print("python", sys.version.split()[0])
for package in ("pandas", "scikit-learn", "scipy", "torch", "transformers", "emoji", "selenium"):
    try:
        print(package, version(package))
    except Exception:
        print(package, "non installato")

## 8. Conclusioni

*Spazio per eventuali considerazioni.*